# Coherence Testing

Note: Run `python -m spacy download en_core_web_sm` before running.


In [1]:
import numpy as np
import re
import string
import spacy
import math
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from coh import compute_heuristic_weights, compute_line_quality, compute_document_quality, prune_dataset

In [2]:
# For perplexity calculations, load a small pretrained model (for demo purposes)
model_name = "gpt2"  # using GPT-2 for demonstration; choose as appropriate
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [12]:
# Example dataset: list of text documents (each document can be multiple lines)
dataset_lines = [
    "This is an example sentence. It is well-formed and clear.",
    "HELLO WORLD! THIS IS A SHOUTING SENTENCE.",
    "the the the repeated words words words.",
    "Sample line with proper punctuation and structure.",
    "lorem ipsum dolor sit amet, consectetur adipiscing elit.",
]


# Read dataset from file and remove special characters
with open('../listbranching/in.txt', 'r', encoding='cp1252') as file:
    dataset_lines = [re.sub(r'[^\w\s]', '', line.strip()) for line in file.readlines()]

dataset_lines = dataset_lines[:10]
dataset_lines


['Once upon a time there was a little',
 'Once upon a time there was a young',
 'Once upon a time there was a girl',
 'Once upon a time there was a man',
 'Once upon a time there was a beautiful',
 'Once upon a time there was an old',
 'Once upon a time there was an island',
 'Once upon a time there was an idea',
 'Once upon a time there was an artist',
 'Once upon a time there was an American']

In [13]:
# Compute weights for each heuristic filter
print("Computing heuristic weights...")
weights = compute_heuristic_weights(dataset_lines, model, tokenizer, device)
for h, w in weights.items():
    print(f"{h}: {w:.4f}")

# Compute and display quality scores for each line in our example dataset
print("\nLine quality scores:")
for line in dataset_lines:
    score = compute_line_quality(line, weights)
    print(f"'{line}' -> {score:.4f}")

# Example document (multi-line string)
document = "\n".join(dataset_lines)
doc_quality = compute_document_quality(document, weights)
print(f"\nDocument quality score: {doc_quality:.4f}")

# For demonstration, assume dataset is a list of documents (here using the same doc multiple times)
dataset_docs = [document] * 10  # pretend we have 10 copies
pruned_docs = prune_dataset(dataset_docs, weights, percentile_threshold=80)
print(f"\nPruned dataset size (top 20% quality): {len(pruned_docs)} out of {len(dataset_docs)}")

Computing heuristic weights...
has_first_letter_caps: 0.0000
no_all_caps: 0.0000
word_repetition_ratio_ge_0_2: 0.0000
digit_punctuation_ratio: 0.0000
no_special_characters: 0.0000
terminal_punctuation: 0.0000
stop_word_match_2: 0.0000
javascript_flag: 0.0000
token_count_ge_3: 0.0000
word_count_3_256: 0.0000
has_object: 0.0000
has_noun: 0.0000
has_determiner: 0.0000
text_complexity_c1: 0.0000

Line quality scores:
'Once upon a time there was a little' -> 0.0000
'Once upon a time there was a young' -> 0.0000
'Once upon a time there was a girl' -> 0.0000
'Once upon a time there was a man' -> 0.0000
'Once upon a time there was a beautiful' -> 0.0000
'Once upon a time there was an old' -> 0.0000
'Once upon a time there was an island' -> 0.0000
'Once upon a time there was an idea' -> 0.0000
'Once upon a time there was an artist' -> 0.0000
'Once upon a time there was an American' -> 0.0000

Document quality score: 0.0000

Pruned dataset size (top 20% quality): 10 out of 10
